In [7]:
!pip install tensorflow keras scikit-learn

Data Processing

In [8]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Step 1: Load Data
def preprocess_selected_params(file_path, seq_len=24):
    # Load data
    df = pd.read_csv(file_path)

    # Simulate CO if not present
    if 'co' not in df.columns:
        np.random.seed(42)
        df['co'] = np.random.normal(loc=0.8, scale=0.3, size=len(df))

    # Drop rows with missing target or required inputs
    df = df.dropna(subset=["pm2.5", "TEMP", "co"])

    # Keep only selected features
    df = df[["co", "TEMP", "pm2.5"]]

    # Normalize
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)

    joblib.dump(scaler, "scaler.pkl")


    # Create time series sequences
    X, y = [], []
    for i in range(len(scaled_data) - seq_len):
        X.append(scaled_data[i:i+seq_len])
        y.append(scaled_data[i+seq_len, 2])  # b/c pm2.5 = column index 2
        #y.append(scaled_data[i+seq_len, 3])  # pm2.5 is at index 3

    return np.array(X), np.array(y)




Model parameters

In [9]:
# Step 3: Process Data
SEQ_LEN = 24
X, y = preprocess_selected_params(file_path="PRSA_data_2010.1.1-2014.12.31.csv", seq_len=24)

# Step 4: Train/Test Split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Step 5: Define and Train LSTM
model = Sequential()
model.add(LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

e:\miniconda\envs\ai_bootcamp\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Train model

In [10]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.1,
          callbacks=[EarlyStopping(patience=3)])

Epoch 1/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.0014 - val_loss: 5.5658e-04
Epoch 2/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6.8561e-04 - val_loss: 4.7377e-04
Epoch 3/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 6.4397e-04 - val_loss: 4.6524e-04
Epoch 4/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 6.3501e-04 - val_loss: 4.5648e-04
Epoch 5/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 6.2983e-04 - val_loss: 4.6159e-04
Epoch 6/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6.2326e-04 - val_loss: 4.5017e-04
Epoch 7/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6.1540e-04 - val_loss: 4.6124e-04
Epoch 8/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6.1735e-04 - val_loss: 4.6727e-04
Epoch 9/10
939/939 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 6.1774e-04 - val_loss: 4.7520e-04


Save model

In [11]:
model.save("lstm_model.keras")

Predict

In [12]:
# Load model
model = load_model("lstm_model.keras")

# Predict again
future_prediction = model.predict(X_test)

NameError: name 'load_model' is not defined

In [ ]:
future_prediction

array([[0.05617213],
       [0.08636937],
       [0.17702842],
       ...,
       [0.00737313],
       [0.0066048 ],
       [0.00439043]], dtype=float32)